# ETL: base de datos NASA Space Apps (Adapting Farms with NASA Data)

**Uso en cada sesión nueva:** ejecuta las celdas de arriba abajo.
Cada tabla se salta sola si su parquet ya está en Drive. Para recalcular una tabla,
pon `FORZAR = {"nombre_tabla"}` en la celda 1.

| # | Tabla | Fuente |
|---|---|---|
| 1 | `cells` | Esqueleto: cuadrícula GLDAS 0.25° + zonas piloto (sin fuente externa) |
| 2 | `coverage` | Derivada: qué fuentes cubren cada celda (se llena sola) |
| 3 | `gaez_actual` | GAEZ v4 Tema 5 (rendimientos y producción reales; .rar local) |
| 4 | `climate_monthly` | GLDAS 2.1 (global) y NLDAS-2 (EE. UU.) vía Earthdata |
| 5 | `climatology` | Derivada de `climate_monthly` |
| 6 | `landcover` | NLCD (MRLC vía pygeohydro; solo EE. UU.) |
| 7 | `stations_validation` | CoCoRaHS (CSV descargados) + `climate_monthly` |
| 8 | `gaez_growing` | GAEZ v4 Tema 2 (módulo descargado del visor) |
| 9 | `gaez_suitability` | GAEZ v4 Tema 4 (módulo descargado del visor) |
| 10 | `crop_traits` | Manual (borrador botánico) + fuentes de plagas por explorar |
| 11 | `indicators_monthly` | Derivada: `climate_monthly` (+ `gaez_growing` y t_min pendientes) |

## 0. Instalación (solo sesión nueva; si Colab pide reiniciar, reinicia y sigue en la celda 1)

In [1]:
!pip -q install earthaccess h5netcdf pyarrow pygeohydro rasterio geopandas rioxarray
!apt-get -qq install -y p7zip-full p7zip-rar > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.


## 1. Configuración, Drive y esqueleto (tablas 1 y 2)

In [2]:
import re, json, shutil, subprocess, warnings, requests
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/nasa_space_apps")   # persistente
DB   = ROOT / "db"                                      # parquet de salida
ZIPS = ROOT / "raw_comprimidos"                         # .rar originales
RAW  = Path("/content/raw")                             # archivos descomprimidos (local, rápido)
for p in (DB, ZIPS, RAW):
    p.mkdir(parents=True, exist_ok=True)

FORZAR = set()          # ejemplo: {"gaez_actual", "climate_monthly"}

def hecho(nombre):
    return (DB / f"{nombre}.parquet").exists() and nombre not in FORZAR

def cargar(nombre):
    return pd.read_parquet(DB / f"{nombre}.parquet")

def guardar(df, nombre):
    df.to_parquet(DB / f"{nombre}.parquet", index=False)
    print(f"OK {nombre}.parquet  {df.shape}")

TABLAS = [
    ("cells",               "Esqueleto: cuadrícula GLDAS 0.25° + zonas piloto"),
    ("coverage",            "Derivada: cobertura de fuentes por celda"),
    ("gaez_actual",         "GAEZ v4 Tema 5 (.rar local)"),
    ("climate_monthly",     "GLDAS 2.1 (global) y NLDAS-2 (EE. UU.), Earthdata"),
    ("climatology",         "Derivada de climate_monthly"),
    ("landcover",           "NLCD (MRLC vía pygeohydro), solo EE. UU."),
    ("stations_validation", "CoCoRaHS (CSV) + climate_monthly"),
    ("gaez_growing",        "GAEZ v4 Tema 2 (módulo del visor)"),
    ("gaez_suitability",    "GAEZ v4 Tema 4 (módulo del visor)"),
    ("crop_traits",         "Manual (borrador) + fuentes de plagas por explorar"),
    ("indicators_monthly",  "Derivada: climate_monthly (+ pendientes)"),
]

def estado():
    for i, (t, fuente) in enumerate(TABLAS, 1):
        ruta = DB / f"{t}.parquet"
        if ruta.exists():
            n = len(pd.read_parquet(ruta))
            print(f"{i:>2}. [OK]        {t:<20} {n:>9} filas | {fuente}")
        else:
            print(f"{i:>2}. [pendiente] {t:<20} {'':>9}       | {fuente}")

# ---------- cuadrícula canónica (GLDAS 0.25°) y zonas piloto ----------
RES = 0.25
NLON = int(360 / RES)

PILOTOS = {                      # nombre: (lat, lon)   <- EDITAR
    "california_central": (38.5, -121.4),
    "iowa":               (42.0,  -93.5),
    "pampas_argentina":   (-33.0, -61.5),
    "bajio_mexico":       (20.9, -101.0),
    "punjab_india":       (30.9,  75.8),
    "ebro_espana":        (41.6,  -0.9),
    "rift_kenia":         (-0.3,  35.3),
}
RADIO_KM = 100

def bbox_from_point(lat, lon, radio_km=RADIO_KM):
    dlat = radio_km / 111.0
    dlon = radio_km / (111.0 * np.cos(np.radians(lat)))
    return (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

def snap(b):
    w, s, e, n = b
    return (np.floor(w / RES) * RES, np.floor(s / RES) * RES,
            np.ceil(e / RES) * RES, np.ceil(n / RES) * RES)

def cell_id(lat, lon):
    i = np.floor((np.asarray(lat) + 90) / RES).astype(int)
    j = np.floor((np.asarray(lon) + 180) / RES).astype(int)
    return i * NLON + j

AOIS = {z: snap(bbox_from_point(*c)) for z, c in PILOTOS.items()}

def tabla_cells():
    filas = []
    for zona, (w, s, e, n) in AOIS.items():
        lats = np.arange(s + RES / 2, n, RES)
        lons = np.arange(w + RES / 2, e, RES)
        LA, LO = np.meshgrid(lats, lons, indexing="ij")
        filas.append(pd.DataFrame({"cell_id": cell_id(LA, LO).ravel(),
                                   "lat": LA.ravel(), "lon": LO.ravel(), "zona": zona}))
    return pd.concat(filas).drop_duplicates("cell_id").reset_index(drop=True)

cells = tabla_cells()
guardar(cells, "cells")                                   # TABLA 1

def registrar_cobertura(fuente, cell_ids):                # TABLA 2 (se llena sola)
    ruta = DB / "coverage.parquet"
    nuevo = pd.DataFrame({"cell_id": cells["cell_id"], "fuente": fuente,
                          "disponible": cells["cell_id"].isin(set(cell_ids))})
    if ruta.exists():
        ant = pd.read_parquet(ruta)
        nuevo = pd.concat([ant[ant["fuente"] != fuente], nuevo])
    nuevo.to_parquet(ruta, index=False)

# ---------- utilidades de rasters ----------
def descomprimir_si_falta(nombre, destino):
    destino = Path(destino)
    if destino.exists() and any(destino.rglob("*.tif")):
        return
    arch = ZIPS / nombre
    assert arch.exists(), f"No encuentro {arch}. Sube el archivo a {ZIPS} y ajusta el nombre."
    destino.mkdir(parents=True, exist_ok=True)
    subprocess.run(["7z", "x", str(arch), f"-o{destino}", "-y"], check=True, stdout=subprocess.DEVNULL)
    print("descomprimido en", destino)

def leer_bloques(src, bbox, agg="sum", cero_es_nodata=True):
    """Lee la ventana del bbox y agrega a la cuadrícula canónica de 0.25°."""
    w, s, e, n = bbox
    rx, ry = src.res
    x0, y0 = src.transform.c, src.transform.f
    c0 = int(round((w - x0) / rx)); c1 = int(round((e - x0) / rx))
    r0 = int(round((y0 - n) / ry)); r1 = int(round((y0 - s) / ry))
    a = src.read(1, window=Window(c0, r0, c1 - c0, r1 - r0)).astype("float64")
    malo = ~np.isfinite(a)
    if src.nodata is not None:
        malo |= (a == src.nodata)
    if cero_es_nodata:
        malo |= (a <= 0)
    f = int(round(RES / rx))
    nr, nc = a.shape[0] // f, a.shape[1] // f
    bl = np.where(malo, np.nan, a)[:nr * f, :nc * f].reshape(nr, f, nc, f)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        v = np.nansum(bl, axis=(1, 3)) if agg == "sum" else np.nanmean(bl, axis=(1, 3))
    lat_c = n - (np.arange(nr) + 0.5) * RES
    lon_c = w + (np.arange(nc) + 0.5) * RES
    LA, LO = np.meshgrid(lat_c, lon_c, indexing="ij")
    return pd.DataFrame({"cell_id": cell_id(LA, LO).ravel(), "lat": LA.ravel(),
                         "lon": LO.ravel(), "valor": v.ravel()})

estado()

Mounted at /content/drive
OK cells.parquet  (564, 4)
 1. [OK]        cells                      564 filas | Esqueleto: cuadrícula GLDAS 0.25° + zonas piloto
 2. [OK]        coverage                  2820 filas | Derivada: cobertura de fuentes por celda
 3. [OK]        gaez_actual              42897 filas | GAEZ v4 Tema 5 (.rar local)
 4. [OK]        climate_monthly           8772 filas | GLDAS 2.1 (global) y NLDAS-2 (EE. UU.), Earthdata
 5. [OK]        climatology               8772 filas | Derivada de climate_monthly
 6. [OK]        landcover                 7110 filas | NLCD (MRLC vía pygeohydro), solo EE. UU.
 7. [OK]        stations_validation       1118 filas | CoCoRaHS (CSV) + climate_monthly
 8. [pendiente] gaez_growing                         | GAEZ v4 Tema 2 (módulo del visor)
 9. [pendiente] gaez_suitability                     | GAEZ v4 Tema 4 (módulo del visor)
10. [OK]        crop_traits                 26 filas | Manual (borrador) + fuentes de plagas por explorar
11. [OK]

## 2. TABLA 3: `gaez_actual` (GAEZ v4 Tema 5, .rar local)
Antes de ejecutar: sube el .rar a `MyDrive/nasa_space_apps/raw_comprimidos/` y pon su nombre en `GAEZ5_RAR`.

In [3]:
if hecho("gaez_actual"):
    ancho = cargar("gaez_actual")
    print("ya existe:", ancho.shape)
else:
    from concurrent.futures import ThreadPoolExecutor

    GAEZ5_RAR = "res06.zip"                      # <- EDITAR
    DEST5 = RAW / "gaez_t5"
    descomprimir_si_falta(GAEZ5_RAR, DEST5)

    cand = [p.parent for p in DEST5.rglob("I") if p.is_dir() and (p.parent / "R").is_dir()]
    assert cand, "No encuentro una carpeta con I y R dentro de " + str(DEST5)
    GAEZ5_DIR = cand[0]
    print("GAEZ5_DIR:", GAEZ5_DIR, sorted(p.name for p in GAEZ5_DIR.iterdir()))

    # catálogo a partir de los archivos
    PAT = re.compile(r"^(?P<crop_code>[a-z0-9]+)_(?P<year>\d{4})_(?P<var>[a-z]+)$", re.I)
    filas, sin_patron = [], []
    for p in sorted(GAEZ5_DIR.rglob("*.tif")):
        m = PAT.match(p.stem)
        if not m:
            sin_patron.append(p.name)
            continue
        filas.append({"ruta": p, "carpeta": p.relative_to(GAEZ5_DIR).parts[0],
                      "crop_code": m["crop_code"].lower(), "year": int(m["year"]), "var": m["var"].lower()})
    loc = pd.DataFrame(filas)
    print("tif con patrón:", len(loc), "| sin patrón:", len(sin_patron))
    print(loc.groupby(["carpeta", "var"]).size().unstack(fill_value=0))

    # nombres de cultivo (una consulta ligera al servidor FAO, guardada en Drive)
    NOMBRES = DB / "nombres_cultivo.csv"
    if not NOMBRES.exists():
        try:
            U = "https://gaez-services.fao.org/server/rest/services/res06/ImageServer/query"
            r = requests.get(U, params={"where": "1=1", "outFields": "Name,crop", "returnGeometry": "false",
                                        "resultRecordCount": 1000, "f": "json"}, timeout=(10, 120))
            cs = pd.DataFrame([f["attributes"] for f in r.json()["features"]])
            cs["crop_code"] = cs["Name"].str.split("_").str[0].str.lower()
            cs[["crop_code", "crop"]].drop_duplicates().to_csv(NOMBRES, index=False)
        except Exception as ex:
            print("No se pudieron obtener nombres de cultivo; se usará el código:", type(ex).__name__)
    loc["crop"] = loc["crop_code"]
    if NOMBRES.exists():
        nom = pd.read_csv(NOMBRES).drop_duplicates("crop_code").rename(columns={"crop": "crop_nombre"})
        loc = loc.merge(nom[["crop_code", "crop_nombre"]], on="crop_code", how="left")
        loc["crop"] = loc["crop_nombre"].fillna(loc["crop_code"])

    SUPPLY = {"I": "Irrigated", "R": "Rainfed", "T": "Total"}      # 'V' (agregados de valor) queda fuera
    VARIABLE = {"har": "area_cosechada_kha", "prd": "produccion_kt"}
    sel = loc[loc["carpeta"].isin(SUPPLY) & loc["var"].isin(VARIABLE)].copy()
    sel["water_supply"] = sel["carpeta"].map(SUPPLY)
    print("rasters a procesar:", len(sel), "(esperado: 324)")

    NODATAS = set()

    def procesar(fila):
        try:
            with rasterio.open(fila["ruta"]) as src:
                NODATAS.add(src.nodata)
                partes = []
                for zona, bbox in AOIS.items():
                    d = leer_bloques(src, bbox, agg="sum", cero_es_nodata=True)
                    d["zona"] = zona
                    partes.append(d)
        except Exception as ex:
            print("falló", fila["ruta"].name, type(ex).__name__, ex)
            return None
        d = pd.concat(partes)
        d["crop"], d["crop_code"] = fila["crop"], fila["crop_code"]
        d["water_supply"], d["year"], d["var"] = fila["water_supply"], fila["year"], VARIABLE[fila["var"]]
        return d

    with ThreadPoolExecutor(8) as ex:
        partes = [p for p in ex.map(procesar, [r for _, r in sel.iterrows()]) if p is not None]
    print("nodata vistos:", NODATAS, "| rasters leídos:", len(partes))

    largo = pd.concat(partes)
    ancho = largo.pivot_table(index=["cell_id", "lat", "lon", "zona", "crop_code", "crop", "water_supply", "year"],
                              columns="var", values="valor", aggfunc="sum").reset_index()
    ancho.columns.name = None
    ancho["rendimiento_t_ha"] = ancho["produccion_kt"] / ancho["area_cosechada_kha"]   # 1000 t / 1000 ha
    ancho = ancho[ancho["area_cosechada_kha"] > 0].reset_index(drop=True)
    GRUPOS = {"Pulses", "Other cereals", "Rest of crops", "Fruits and nuts", "Vegetables",
              "Fodder crops", "Stimulants", "Oil seeds", "Root crops", "Cereals", "Main crops"}
    ancho["es_grupo"] = ancho["crop"].isin(GRUPOS)        # marca agregados para no sumarlos con cultivos individuales

    guardar(ancho, "gaez_actual")
    registrar_cobertura("GAEZ_T5", ancho["cell_id"].unique())
    display(ancho.head())

ya existe: (42897, 12)


### 2b. (Opcional) Control de calidad de `gaez_actual`
Solo funciona en la sesión en que se ejecutó la celda anterior (necesita `loc`).
Compara el rendimiento calculado con el raster `yld` de GAEZ.

In [4]:
if "loc" not in globals():
    print("Ejecuta la celda de gaez_actual con FORZAR = {'gaez_actual'} para disponer de 'loc'.")
else:
    cand_qa = ancho[(ancho["water_supply"] == "Rainfed") & (ancho["year"] == 2010)]
    fila = cand_qa.sort_values("area_cosechada_kha").iloc[-1]
    print(fila[["zona", "crop", "lat", "lon", "area_cosechada_kha", "rendimiento_t_ha"]].to_dict())
    ruta = loc[(loc["crop_code"] == fila["crop_code"]) & (loc["carpeta"] == "R") &
               (loc["year"] == 2010) & (loc["var"] == "yld")].iloc[0]["ruta"]
    with rasterio.open(ruta) as src:
        c0 = int(round((fila["lon"] - RES / 2 - src.transform.c) / src.res[0]))
        r0 = int(round((src.transform.f - (fila["lat"] + RES / 2)) / src.res[1]))
        f = int(round(RES / src.res[0]))
        bloque = src.read(1, window=Window(c0, r0, f, f)).astype(float)
    bloque[bloque <= 0] = np.nan
    print("rendimiento GAEZ en los píxeles de la celda (t/ha):\n", np.round(bloque, 2))
    print("media simple:", round(float(np.nanmean(bloque)), 2), "| el nuestro debe caer dentro del rango")

Ejecuta la celda de gaez_actual con FORZAR = {'gaez_actual'} para disponer de 'loc'.


### 2c. (Opcional) Qué es la carpeta V (valores agregados)

In [5]:
try:
    U = "https://gaez-services.fao.org/server/rest/services/res06/ImageServer/query"
    r = requests.get(U, params={"where": "Name LIKE '%_val'", "outFields": "Name,variable,units,file_description",
                                "returnGeometry": "false", "f": "json"}, timeout=(10, 90))
    pd.set_option("display.max_colwidth", 200, "display.width", 250)
    display(pd.DataFrame([f["attributes"] for f in r.json()["features"]]))
except Exception as ex:
    print("No se pudo consultar el servidor:", type(ex).__name__)

,OBJECTID,Name,variable,units,file_description
0,985,all_2000_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2000 for Main crops under Total water supply conditions
1,987,cer_2000_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2000 for Cereals under Total water supply conditions
2,989,oil_2000_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2000 for Oil seeds under Total water supply conditions
3,990,rts_2000_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2000 for Root crops under Total water supply conditions
4,991,all_2010_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2010 for Main crops under Total water supply conditions
5,993,cer_2010_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2010 for Cereals under Total water supply conditions
6,995,oil_2010_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2010 for Oil seeds under Total water supply conditions
7,996,rts_2010_val,Total crop production volume,1000 GK$,Total crop production volume for the year 2010 for Root crops under Total water supply conditions


## 3. TABLAS 4 y 5: `climate_monthly` y `climatology` (GLDAS global + NLDAS EE. UU.)
Primero con **un año** de prueba; cuando los nombres y unidades impresos sean correctos,
cambia `PERIODO_CLIMA` a `("2000-01-01", "2024-12-31")` y recalcula con `FORZAR`.
Con un solo año la climatología no tiene desviación, así que los z-scores saldrán vacíos (es normal).

In [6]:
if hecho("climate_monthly") and hecho("climatology"):
    print("climate_monthly y climatology ya existen")
else:

    import os, getpass
    import earthaccess, xarray as xr

    # Login con token de Earthdata: 1) Secretos de Colab (EARTHDATA_TOKEN), 2) variable de entorno, 3) lo pide aquí
    try:
        from google.colab import userdata
        token = userdata.get("EARTHDATA_TOKEN")
    except Exception:
        token = None
    if not token:
        token = os.environ.get("EARTHDATA_TOKEN") or getpass.getpass("Pega tu token de Earthdata: ")
    os.environ["EARTHDATA_TOKEN"] = token.strip()
    auth = earthaccess.login(strategy="environment")
    print("Earthdata autenticado:", bool(auth.authenticated))

    PERIODO_CLIMA = ("2020-01-01", "2020-12-31")      # PRUEBA; luego ("2000-01-01", "2024-12-31")
    BASE_CLIM = (2000, 2024)

    def construir_clima(fuente, short_name, version, mapa, temporal):
        gr = earthaccess.search_data(short_name=short_name, version=version, temporal=temporal)
        gr = [g for g in gr if any(str(l).endswith((".nc4", ".nc")) for l in g.data_links())]
        print(fuente, short_name, "->", len(gr), "gránulos")
        if not gr:
            return pd.DataFrame()
        ds = xr.open_mfdataset(earthaccess.open(gr), engine="h5netcdf", combine="by_coords").sortby("lat")
        print("variables disponibles:", list(ds.data_vars))
        elegidas = {c: next((v for v in cands if v in ds.data_vars), None) for c, cands in mapa.items()}
        print("mapeo:", elegidas)
        elegidas = {c: v for c, v in elegidas.items() if v}
        unidades = {c: ds[v].attrs.get("units", "") for c, v in elegidas.items()}
        print("unidades:", unidades)

        partes = []
        for zona, (w, s, e, n) in AOIS.items():
            sub = ds[list(elegidas.values())].sel(lat=slice(s, n), lon=slice(w, e))
            if sub.sizes["lat"] == 0 or sub.sizes["lon"] == 0:
                continue                      # la zona queda fuera de la cobertura de la fuente
            d = sub.load().to_dataframe().reset_index().rename(columns={v: c for c, v in elegidas.items()})
            d["zona"] = zona
            partes.append(d)
        if not partes:
            return pd.DataFrame()
        d = pd.concat(partes)
        d = d.dropna(subset=list(elegidas), how="all")
        d["time"] = pd.to_datetime(d["time"])
        d["cell_id"] = cell_id(d["lat"], d["lon"])

        dias = d["time"].dt.days_in_month
        for c in ("precip", "evap"):
            if c in d and "s-1" in unidades.get(c, "").replace(" ", ""):
                d[c] = d[c] * 86400 * dias                    # kg m-2 s-1 -> mm/mes
        for c, espesor_mm in (("sm_0_10", 100), ("sm_10_40", 300), ("sm_40_100", 600)):
            if c in d and "kg" in unidades.get(c, ""):
                d[c] = d[c] / espesor_mm                      # kg m-2 (mm de agua) -> fracción volumétrica
        if "tair" in d and d["tair"].median() > 150:
            d["tair"] = d["tair"] - 273.15                    # K -> °C

        d["año"], d["mes"] = d["time"].dt.year, d["time"].dt.month
        cols = [c for c in ("sm_0_10", "sm_10_40", "sm_40_100", "precip", "evap", "tair") if c in d]
        out = (d.groupby(["zona", "cell_id", "año", "mes"], as_index=False)
                 .agg({**{c: "mean" for c in cols}, "lat": "mean", "lon": "mean"}))
        out["fuente"] = fuente
        return out

    MAPA_GLDAS = {"sm_0_10": ["SoilMoi0_10cm_inst"], "sm_10_40": ["SoilMoi10_40cm_inst"],
                  "sm_40_100": ["SoilMoi40_100cm_inst"], "precip": ["Rainf_f_tavg", "Rainf_tavg"],
                  "evap": ["Evap_tavg"], "tair": ["Tair_f_inst"]}
    gl = construir_clima("GLDAS", "GLDAS_NOAH025_M", "2.1", MAPA_GLDAS, PERIODO_CLIMA)

    MAPA_NOAH = {"sm_0_10": ["SoilM_0_10cm"], "sm_10_40": ["SoilM_10_40cm"],
                 "sm_40_100": ["SoilM_40_100cm"], "evap": ["Evap"]}
    MAPA_FORA = {"precip": ["APCP", "Rainf"], "tair": ["Tair"]}
    nl = pd.DataFrame()
    try:
        nl_a = construir_clima("NLDAS", "NLDAS_NOAH0125_M", "2.0", MAPA_NOAH, PERIODO_CLIMA)
        nl_b = construir_clima("NLDAS", "NLDAS_FORA0125_M", "2.0", MAPA_FORA, PERIODO_CLIMA)
        llave = ["fuente", "zona", "cell_id", "año", "mes"]
        if len(nl_a) and len(nl_b):
            nl = nl_a.merge(nl_b.drop(columns=["lat", "lon"]), on=llave, how="outer")
        else:
            print("NLDAS incompleto (revisa short_name/versión con la consulta a CMR); se omite")
    except Exception as ex:
        print("NLDAS falló:", type(ex).__name__, ex)

    cm = pd.concat([x for x in (gl, nl) if len(x)], ignore_index=True)
    assert len(cm), "Sin datos de clima: revisa short_name, versión y login de Earthdata"
    if "precip" in cm and "evap" in cm:
        cm["balance"] = cm["precip"] - cm["evap"]            # mm/mes; negativo = déficit
    cm = cm.sort_values(["fuente", "cell_id", "año", "mes"]).reset_index(drop=True)

    VARS = [v for v in ("sm_0_10", "sm_10_40", "sm_40_100", "precip", "evap", "tair", "balance") if v in cm]
    b = cm[cm["año"].between(*BASE_CLIM)]
    g = b.groupby(["fuente", "cell_id", "mes"])[VARS]
    clim = pd.concat({"media": g.mean(), "desv": g.std(), "p10": g.quantile(.10), "p90": g.quantile(.90)}, axis=1)
    clim.columns = [f"{v}_{s}" for s, v in clim.columns]
    clim = clim.reset_index()

    m = cm[["fuente", "cell_id", "mes"]].merge(clim, on=["fuente", "cell_id", "mes"], how="left")
    for v in VARS:
        z = (cm[v].values - m[f"{v}_media"].values) / m[f"{v}_desv"].values
        cm[f"{v}_z"] = np.where(np.isfinite(z), z, np.nan)

    guardar(cm, "climate_monthly")
    guardar(clim, "climatology")
    for f in cm["fuente"].unique():
        registrar_cobertura(f, cm.loc[cm["fuente"] == f, "cell_id"].unique())
    display(cm.groupby(["fuente", "zona"]).size().unstack(0))

climate_monthly y climatology ya existen


## 4. TABLA 6: `landcover` (NLCD, solo zonas de EE. UU.)
`pygeohydro` da el NLCD "clásico" (2001-2021), no la serie anual desde 1985.

In [7]:
if hecho("landcover"):
    print("landcover ya existe")
else:
    from pygeohydro import nlcd_bygeom
    from shapely.geometry import box
    import geopandas as gpd

    NLCD_CLASSES = {11: "Agua", 12: "Hielo/nieve", 21: "Desarrollado (abierto)", 22: "Desarrollado (bajo)",
                    23: "Desarrollado (medio)", 24: "Desarrollado (alto)", 31: "Suelo desnudo",
                    41: "Bosque caducifolio", 42: "Bosque perennifolio", 43: "Bosque mixto",
                    52: "Matorral", 71: "Pastizal", 81: "Pasto/heno", 82: "Cultivos",
                    90: "Humedal leñoso", 95: "Humedal herbáceo"}
    NLCD_ANIOS, NLCD_RES = [2001, 2011, 2021], 120          # metros; baja a 30 si el servicio lo aguanta

    def en_conus(lat, lon):
        return 24.5 <= lat <= 49.5 and -125 <= lon <= -66.5

    ZONAS_US = {z: AOIS[z] for z, c in PILOTOS.items() if en_conus(*c)}
    print("zonas con NLCD:", list(ZONAS_US))

    filas = []
    for zona, (w, s, e, n) in ZONAS_US.items():
        try:
            res = nlcd_bygeom(gpd.GeoSeries([box(w, s, e, n)], crs=4326), resolution=NLCD_RES,
                              years={"cover": NLCD_ANIOS})
        except Exception as ex:
            print("NLCD falló en", zona, type(ex).__name__, ex)
            continue
        ds = res[0] if isinstance(res, dict) else res
        print(zona, list(ds.data_vars))
        for y in NLCD_ANIOS:
            da = ds[f"cover_{y}"]
            X, Y = np.meshgrid(da["x"].values, da["y"].values)
            if np.nanmax(np.abs(X)) > 360:                   # coordenadas proyectadas -> lon/lat
                from pyproj import Transformer
                try:
                    crs_in = ds.rio.crs
                except Exception:
                    crs_in = None
                X, Y = Transformer.from_crs(crs_in or 5070, 4326, always_xy=True).transform(X, Y)
            v = da.values.astype(float)
            ok = np.isfinite(v) & (v > 0)
            d = pd.DataFrame({"cell_id": cell_id(Y[ok], X[ok]), "clase_id": v[ok].astype(int)})
            t = d.groupby(["cell_id", "clase_id"]).size().rename("px").reset_index()
            t["pct"] = 100 * t["px"] / t.groupby("cell_id")["px"].transform("sum")
            t["año"], t["zona"] = y, zona
            filas.append(t)

    if filas:
        lc = pd.concat(filas)
        lc["clase"] = lc["clase_id"].map(NLCD_CLASSES).fillna("Otra")
        lc["es_cultivo"] = lc["clase_id"].isin([81, 82])
        lc = lc[["zona", "cell_id", "año", "clase_id", "clase", "es_cultivo", "pct"]]
        guardar(lc, "landcover")
        registrar_cobertura("NLCD", lc["cell_id"].unique())
        display(lc.groupby(["zona", "año", "clase"])["pct"].sum().round(1).head(30))
    else:
        print("landcover sin datos: revisa los errores de arriba")

landcover ya existe


## 5. TABLA 7: `stations_validation` (lluvia de estaciones CoCoRaHS vs. modelo)
Fuente por defecto: **GHCN-Daily de NOAA/NCEI**, que incluye estaciones CoCoRaHS (ID con prefijo `US1`).
Las estaciones se buscan en el inventario de GHCN-Daily y su lluvia se pide al NCEI Access Data Service.
La API directa de CoCoRaHS (`FUENTE_COCO = "cocorahs"`) respondió con una página HTML en vez de datos.
Los datos se guardan en `raw_cocorahs/` (caché). Edita `ZONAS_COCO` y `PERIODO_COCO`; para la prueba usa el mismo año que el clima.

In [8]:
import io, time
COCO = ROOT / "raw_cocorahs"
COCO.mkdir(exist_ok=True)

ZONAS_COCO = {"california_central": "CA", "iowa": "IA"}      # zona piloto de EE. UU. -> estado   <- EDITAR
PERIODO_COCO = ("2020-01-01", "2020-12-31")                  # PRUEBA; luego el mismo rango que el clima
MAX_ESTACIONES = 40                                          # estaciones más cercanas al centro de la zona
FUENTE_COCO = "ncei"                                         # "ncei" (recomendada) o "cocorahs" (API directa)
UNIDAD_LLUVIA = "in"                                         # solo aplica a FUENTE_COCO = "cocorahs"

NCEI    = "https://www.ncei.noaa.gov/access/services/data/v1"
EXP_ST  = "https://data.cocorahs.org/cocorahs/export/exportstations.aspx"
EXP_REP = "https://data.cocorahs.org/export/exportreports.aspx"

def _tabla_remota(url, params):
    err = None
    for intento in range(3):
        try:
            r = requests.get(url, params=params, timeout=(15, 180))
            if r.status_code >= 400:
                err = ValueError(f"HTTP {r.status_code}: {r.text[:300]!r}")
                if r.status_code < 500:
                    break                                  # error del cliente: reintentar no sirve
                time.sleep(2 * (intento + 1))
                continue
            txt = r.text.strip()
            if not txt:
                return pd.DataFrame()
            if txt.startswith("<"):
                raise ValueError("la respuesta no son datos. Primeros caracteres: " + txt[:200].replace("\n", " "))
            try:
                dato = r.json()
                if isinstance(dato, dict):
                    dato = next((v for v in dato.values() if isinstance(v, list)), [dato])
                return pd.json_normalize(dato)
            except ValueError:
                return pd.read_csv(io.StringIO(r.text))
        except Exception as ex:
            err = ex
            time.sleep(2 * (intento + 1))
    print("falló la consulta:", url, params, "|", type(err).__name__, str(err)[:400])
    return pd.DataFrame()

def _col(df, *patrones):
    for p in patrones:
        for c in df.columns:
            if re.search(p, str(c), re.I):
                return c
    return None

def _mas_cercanas(est, zona):
    la0, lo0 = PILOTOS[zona]
    est = est.copy()
    est["dist"] = np.hypot(est["lat"] - la0, (est["lon"] - lo0) * np.cos(np.radians(la0)))
    return est.sort_values("dist").head(MAX_ESTACIONES).drop(columns="dist")

INV_URL = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-inventory.txt"

def inventario_us1():
    """Estaciones CoCoRaHS (prefijo US1) con PRCP en GHCN-Daily: id, lat, lon, años de inicio y fin."""
    cache = COCO / "ghcnd_us1_prcp.csv"
    if cache.exists():
        return pd.read_csv(cache)
    bruto = COCO / "ghcnd-inventory.txt"
    if not bruto.exists():
        r = requests.get(INV_URL, timeout=(15, 300))
        r.raise_for_status()
        bruto.write_bytes(r.content)
    inv = pd.read_fwf(bruto, colspecs=[(0, 11), (12, 20), (21, 30), (31, 35), (36, 40), (41, 45)],
                      names=["estacion", "lat", "lon", "elemento", "anio_ini", "anio_fin"])
    inv = inv[(inv["elemento"] == "PRCP") & inv["estacion"].astype(str).str.startswith("US1")]
    inv = inv.drop(columns="elemento")
    inv.to_csv(cache, index=False)
    return inv

def descargar_ncei(zona, ini, fin, lote=20):
    """1) estaciones CoCoRaHS (US1) de la zona con datos en el periodo, desde el inventario de GHCN-Daily;
       2) PRCP diaria (mm) de esas estaciones desde el NCEI Access Data Service."""
    inv = inventario_us1()
    w, s, e, n = AOIS[zona]
    ok = (inv["lat"].between(s, n) & inv["lon"].between(w, e) &
          (inv["anio_ini"] <= int(ini[:4])) & (inv["anio_fin"] >= int(fin[:4])))
    est = inv.loc[ok, ["estacion", "lat", "lon"]]
    print(zona, "| estaciones CoCoRaHS (US1) en la zona con datos en el periodo:", len(est))
    if est.empty:
        return pd.DataFrame(), pd.DataFrame()
    est = _mas_cercanas(est, zona)
    ids = est["estacion"].tolist()
    partes = []
    for y in range(int(ini[:4]), int(fin[:4]) + 1):
        a = max(pd.Timestamp(ini), pd.Timestamp(f"{y}-01-01")).strftime("%Y-%m-%d")
        b = min(pd.Timestamp(fin), pd.Timestamp(f"{y}-12-31")).strftime("%Y-%m-%d")
        for i in range(0, len(ids), lote):
            p = {"dataset": "daily-summaries", "dataTypes": "PRCP", "stations": ",".join(ids[i:i + lote]),
                 "startDate": a, "endDate": b, "format": "csv", "units": "metric"}
            tbl = _tabla_remota(NCEI, p)
            if len(tbl):
                partes.append(tbl)
            time.sleep(0.5)
    if not partes:
        return est, pd.DataFrame()
    rep = pd.concat(partes, ignore_index=True)
    print(zona, "| columnas NCEI:", list(rep.columns), "| filas:", len(rep))
    return est, rep

def estaciones_cocorahs(zona, estado):
    est = pd.DataFrame()
    for pais in (None, "usa", "USA"):
        p = {"format": "json", "state": estado}
        if pais:
            p["country"] = pais
        est = _tabla_remota(EXP_ST, p)
        if len(est):
            break
    if est.empty:
        return est
    c_id = _col(est, r"^(st_?num|station_?number|station_?id)$", r"station.*(num|id)")
    c_lat, c_lon = _col(est, r"^lat"), _col(est, r"^(lng|lon)")
    print(zona, "| columnas de estaciones:", list(est.columns)[:12], "->", c_id, c_lat, c_lon)
    if None in (c_id, c_lat, c_lon):
        return pd.DataFrame()
    est = est.rename(columns={c_id: "estacion", c_lat: "lat", c_lon: "lon"})[["estacion", "lat", "lon"]].copy()
    est["lat"] = pd.to_numeric(est["lat"], errors="coerce")
    est["lon"] = pd.to_numeric(est["lon"], errors="coerce")
    w, s, e, n = AOIS[zona]
    est = est.dropna()
    est = est[est["lat"].between(s, n) & est["lon"].between(w, e)]
    return _mas_cercanas(est, zona)

def reportes_cocorahs(est, ini, fin, lote=20):
    ids = est["estacion"].astype(str).tolist()
    partes = []
    for y in range(int(ini[:4]), int(fin[:4]) + 1):
        a = max(pd.Timestamp(ini), pd.Timestamp(f"{y}-01-01"))
        b = min(pd.Timestamp(fin), pd.Timestamp(f"{y}-12-31"))
        for i in range(0, len(ids), lote):
            p = {"Format": "csv", "ReportType": "Daily", "ResponseFields": "all",
                 "ReportDateType": "reportdate", "StartDate": a.strftime("%m/%d/%Y"),
                 "EndDate": b.strftime("%m/%d/%Y"), "stations": ",".join(ids[i:i + lote])}
            t = _tabla_remota(EXP_REP, p)
            if len(t):
                partes.append(t)
            time.sleep(0.5)
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

def construir_validacion(rep, est, clima, unidad="in"):
    c_id = _col(rep, r"^stationnumber$", r"station.*(num|id)", r"^station$")
    c_fe = _col(rep, r"^observationdate$", r"obs.*date", r"^date$")
    c_ll = _col(rep, r"^totalprecipamt$", r"totalprecip", r"precip.*amt", r"^prcp$", r"gauge")
    print("columnas detectadas en reportes (estación, fecha, lluvia):", c_id, c_fe, c_ll)
    if None in (c_id, c_fe, c_ll):
        print("No pude detectar las columnas. Columnas disponibles:", list(rep.columns))
        return pd.DataFrame()
    lluvia = rep[c_ll].astype(str).str.strip().replace({"T": "0", "NA": np.nan, "": np.nan, "nan": np.nan})
    r = pd.DataFrame({"estacion": rep[c_id].astype(str).str.strip(),
                      "fecha": pd.to_datetime(rep[c_fe], errors="coerce"),
                      "lluvia": pd.to_numeric(lluvia, errors="coerce")}).dropna()
    print("lluvia diaria cruda (min/mediana/máx):", r["lluvia"].min(), r["lluvia"].median(), r["lluvia"].max())
    if unidad == "in":
        r["lluvia"] = r["lluvia"] * 25.4
    r = r.merge(est[["estacion", "lat", "lon"]].astype({"estacion": str}).drop_duplicates("estacion"),
                on="estacion", how="inner")
    r["año"], r["mes"], r["dias"] = r["fecha"].dt.year, r["fecha"].dt.month, r["fecha"].dt.days_in_month
    mes = (r.groupby(["estacion", "lat", "lon", "año", "mes"], as_index=False)
             .agg(lluvia_estacion_mm=("lluvia", "sum"), n_obs=("lluvia", "size"), dias=("dias", "first")))
    mes = mes[mes["n_obs"] >= 0.8 * mes["dias"]].drop(columns="dias")      # meses con >=80% de días reportados
    mes["cell_id"] = cell_id(mes["lat"], mes["lon"])
    mod = clima[["fuente", "cell_id", "año", "mes", "precip"]].rename(
        columns={"fuente": "fuente_modelo", "precip": "precip_modelo_mm"})
    return mes.merge(mod, on=["cell_id", "año", "mes"], how="inner")

if hecho("stations_validation"):
    print("stations_validation ya existe")
elif not hecho("climate_monthly"):
    print("Primero genera climate_monthly (la validación compara contra esa tabla)")
else:
    sv_partes = []
    clima_cm = cargar("climate_monthly")
    for zona, estado in ZONAS_COCO.items():
        ae, ar = COCO / f"{FUENTE_COCO}_stations_{zona}.csv", COCO / f"{FUENTE_COCO}_reports_{zona}.csv"
        if ae.exists() and ar.exists():
            est, rep = pd.read_csv(ae), pd.read_csv(ar)
            print(zona, "| desde caché:", len(est), "estaciones,", len(rep), "reportes")
        else:
            if FUENTE_COCO == "ncei":
                est, rep = descargar_ncei(zona, *PERIODO_COCO)
            else:
                est = estaciones_cocorahs(zona, estado)
                rep = reportes_cocorahs(est, *PERIODO_COCO) if len(est) else pd.DataFrame()
            if est.empty or rep.empty:
                print("sin datos para", zona); continue
            est.to_csv(ae, index=False); rep.to_csv(ar, index=False)
            print(zona, "| descargado:", len(est), "estaciones,", len(rep), "reportes")
        unidad = "mm" if FUENTE_COCO == "ncei" else UNIDAD_LLUVIA
        v = construir_validacion(rep, est, clima_cm, unidad)
        if len(v):
            v["zona"] = zona
            sv_partes.append(v)
    if sv_partes:
        sv = pd.concat(sv_partes, ignore_index=True)
        guardar(sv, "stations_validation")
        registrar_cobertura("CoCoRaHS", sv["cell_id"].unique())
        resumen = sv.groupby("fuente_modelo").apply(lambda x: pd.Series({
            "meses": len(x), "correlacion": x["lluvia_estacion_mm"].corr(x["precip_modelo_mm"]),
            "sesgo_mm": (x["precip_modelo_mm"] - x["lluvia_estacion_mm"]).mean()}))
        display(resumen)
    else:
        print("No se generó stations_validation: revisa los mensajes de arriba")

stations_validation ya existe


## 6. TABLAS 8 y 9: `gaez_growing` (Tema 2) y `gaez_suitability` (Tema 4), leídas directo por URL
Estos temas no se descargan a mano: el catálogo del servidor GAEZ da, para cada raster, un `download_url` (GeoTIFF en S3),
y se lee **solo la ventana de cada zona piloto** sin bajar el archivo. Dos pasos:
1. **6a (descubrimiento):** lista los servicios, y muestra qué valores tiene el catálogo (variable, cultivo, escenario...).
   Pega la salida para decidir los filtros.
2. **6b (construcción):** con `SERV_T2/SERV_T4` y `WHERE_T2/WHERE_T4` definidos, construye las tablas.


In [20]:
import time
from concurrent.futures import ThreadPoolExecutor

GAEZ = "https://gaez-services.fao.org/server/rest/services"

def _gaez(url, params, post=True):
    p = {**params, "f": "json"}
    err = None
    for intento in range(3):
        try:
            r = requests.post(url, data=p, timeout=(15, 180)) if post else requests.get(url, params=p, timeout=(15, 180))
            r.raise_for_status()
            j = r.json()
            if "error" in j:
                raise RuntimeError(j["error"])
            return j
        except Exception as ex:
            err = ex
            time.sleep(2 * (intento + 1))
    raise err

def campos_ligeros(servicio):
    info = _gaez(f"{GAEZ}/{servicio}/ImageServer", {}, post=False)
    return ",".join(f["name"] for f in info["fields"]
                    if "Blob" not in f["type"] and "Geometry" not in f["type"])

def ids_servicio(servicio, where="1=1"):
    j = _gaez(f"{GAEZ}/{servicio}/ImageServer/query", {"where": where, "returnIdsOnly": "true"})
    return list(j.get("objectIds") or [])

def registros(servicio, ids, campos, lote=300):
    filas = []
    for i in range(0, len(ids), lote):
        j = _gaez(f"{GAEZ}/{servicio}/ImageServer/query",
                  {"objectIds": ",".join(map(str, ids[i:i + lote])), "outFields": campos, "returnGeometry": "false"})
        filas += [f["attributes"] for f in j.get("features", [])]
    return pd.DataFrame(filas)

def listar_servicios():
    j = _gaez(GAEZ, {}, post=False)
    out = [(s["name"], s["type"]) for s in j.get("services", [])]
    for carp in j.get("folders", []):
        jj = _gaez(f"{GAEZ}/{carp}", {}, post=False)
        out += [(s["name"], s["type"]) for s in jj.get("services", [])]
    return out

def resumir_catalogo(servicio, n=1500, mostrar=25):
    """Muestra repartida del catálogo (ids espaciados) y valores distintos por campo de texto."""
    ids = ids_servicio(servicio)
    print(f"{servicio}: {len(ids)} rasters en el catálogo")
    if not ids:
        return pd.DataFrame()
    paso = max(1, len(ids) // n)
    m = registros(servicio, ids[::paso], campos_ligeros(servicio))
    print("campos:", list(m.columns), "| ¿tiene download_url?:", "download_url" in m.columns)
    for c in m.columns:
        if c in ("download_url", "filepath", "renderer", "file_id", "file_description", "Shape") or m[c].dtype != object:
            continue
        print(f"\n{c} ({m[c].nunique()} valores en la muestra):", m[c].value_counts().head(mostrar).to_dict())
    return m

# ---- DESCUBRIMIENTO: ejecuta esto y pega la salida ----
try:
    servicios = listar_servicios()
    imgs = [n for n, t in servicios if t == "ImageServer"]
    print("ImageServers disponibles:")
    for n in imgs:
        try:
            info = _gaez(f"{GAEZ}/{n}/ImageServer", {}, post=False)
            desc = (info.get("serviceDescription") or info.get("description") or "")[:120].replace("\n", " ")
            cnt = _gaez(f"{GAEZ}/{n}/ImageServer/query", {"where": "1=1", "returnCountOnly": "true"})["count"]
            print(f"  {n:<16} {cnt:>8} rasters | {desc}")
        except Exception as ex:
            print(f"  {n:<16} (no se pudo leer: {type(ex).__name__})")
except Exception as ex:
    print("No pude listar los servicios:", type(ex).__name__, str(ex)[:200])

SERV_T4 = "res05"                 # Tema 4 (idoneidad y rendimiento alcanzable), según la URL que pegaste
muestra_t4 = resumir_catalogo(SERV_T4)

ImageServers disponibles:
  LR                    134 rasters | LR
  res01              111606 rasters | res01
  res02              111672 rasters | res02
  res05              122708 rasters | res05
  res06                 498 rasters | res06
  res07                 288 rasters | res07
res05: 122708 rasters en el catálogo
campos: ['objectid', 'name', 'category', 'filepath', 'sub_theme_name', 'variable', 'file_description', 'year', 'model', 'rcp', 'crop', 'water_supply', 'input_level', 'c02_fertilization', 'units', 'renderer', 'download_url', 'file_id'] | ¿tiene download_url?: True

name (1184 valores en la muestra): {'sx2Hr0_csv': 4, 'suHi_cit': 4, 'wdlHr_wpo': 4, 'ylHr0_tob': 4, 'siHr0_cab': 3, 'scHr_cof': 3, 'scHi_csv': 3, 'sx1Hr_fml': 3, 'scHr0_cot': 3, 'etlHr_spo': 3, 'etcHr_alf': 3, 'etcHi0_cot': 3, 'yxHr0_oni': 3, 'siHr_rye': 3, 'ycHr0_jtr': 3, 'yxHr_spo': 3, 'sx2Hr0_soy': 3, 'scHr_swg': 3, 'scHr0_brl': 3, 'scHr_rsd': 3, 'ycHr0_ban': 3, 'suHi0_csv': 3, 'etlHr0_whe': 3, 'etcHr0_co

### 6a-bis. Descubrimiento del Tema 2 (hipótesis: `res01`) y de `LR` (recursos de suelo y agua)
`res01` y `res02` tienen ~111.600 rasters cada uno; no sé cuál es el Tema 2. Esta celda resume `res01`.
`LR` solo tiene 134 rasters: se muestran los atributos para ver si trae suelo y relieve (posible tabla extra).

In [21]:
muestra_t2 = resumir_catalogo("res01", mostrar=15)

res01: 111606 rasters en el catálogo
campos: ['objectid', 'name', 'category', 'filepath', 'sub_theme_name', 'variable', 'file_description', 'year', 'model', 'rcp', 'units', 'renderer', 'download_url', 'file_id'] | ¿tiene download_url?: True

name (1509 valores en la muestra): {'KG2_CRUTS32_Hist_0010': 1, 'mc2_CRUTS32_Hist_7100': 1, 'rQ1_CRUTS32_Hist_0010': 1, 'ts3_CRUTS32_Hist_7100': 1, 'Wb_1965': 1, 'cbt_1988': 1, 'epx_1960': 1, 'et0_1983': 1, 'eta_2006': 1, 'fst_1978': 1, 'lac_2001': 1, 'lgb_1973': 1, 'lgd_1996': 1, 'lt2_1968': 1, 'lt3_1991': 1}

sub_theme_name (4 valores en la muestra): {'Moisture Regime': 706, 'Thermal Regime': 465, 'Growing Period': 255, 'Climate Classification': 82}

variable (56 valores en la muestra): {'Seasonal P/PET ratio (*100) in winter (October-March in northern hemisphere)': 54, 'Quarterly P/PET ratio (July-September)': 54, 'Net primary production (with irrigation)': 51, 'Net primary production (rain-fed)': 49, 'Quarterly P/PET ratio (April-June)': 49, 'Q

In [22]:
ids_lr = ids_servicio("LR")
lr = registros("LR", ids_lr, campos_ligeros("LR"))
print("LR:", len(lr), "rasters | columnas:", list(lr.columns))
cols_lr = [c for c in lr.columns if c.lower() in ("name", "sub_theme_name", "variable", "units", "year")]
pd.set_option("display.max_rows", 200, "display.width", 250, "display.max_colwidth", 80)
display(lr[cols_lr].head(60))

LR: 134 rasters | columnas: ['OBJECTID', 'Name', 'Category', 'sub_theme_name', 'variable', 'file_description', 'year', 'model', 'rcp', 'units', 'renderer', 'download_url', 'file_id']


,Name,sub_theme_name,variable,year,units
0,aez_v9v2red_5m_CRUTS32_Hist_8110_100_avg,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,1981-2010,Class
1,aez_v9v2red_5m_ENSEMBLE_rcp2p6_2020s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2011-2040,Class
2,aez_v9v2red_5m_ENSEMBLE_rcp2p6_2050s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2041-2070,Class
3,aez_v9v2red_5m_ENSEMBLE_rcp2p6_2080s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2071-2100,Class
4,aez_v9v2red_5m_ENSEMBLE_rcp4p5_2020s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2011-2040,Class
5,aez_v9v2red_5m_ENSEMBLE_rcp4p5_2050s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2041-2070,Class
6,aez_v9v2red_5m_ENSEMBLE_rcp4p5_2080s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2071-2100,Class
7,aez_v9v2red_5m_ENSEMBLE_rcp6p0_2020s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2011-2040,Class
8,aez_v9v2red_5m_ENSEMBLE_rcp6p0_2050s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2041-2070,Class
9,aez_v9v2red_5m_ENSEMBLE_rcp6p0_2080s,Agro-ecological Zones,Dominant AEZ class (33 classes) at 5 arc-minutes,2071-2100,Class


### 6a-ter. Variables HISTÓRICAS del Tema 2 (`res01`), por subtema
Lista cada variable histórica con su subtema, unidades, nº de rasters, cuántos son años sueltos y qué periodos agregados existen.
Con esto se escribe el filtro de `gaez_growing` (periodo de crecimiento, regímenes térmico y de humedad).

In [23]:
ids_h = ids_servicio("res01", "rcp = 'Historical'")
print("rasters históricos en res01:", len(ids_h))
h = registros("res01", ids_h, "objectid,name,sub_theme_name,variable,year,model,rcp,units")
h["sub_theme_name"] = h["sub_theme_name"].str.strip()
h["anual"] = h["year"].astype(str).str.fullmatch(r"\d{4}")
resumen_h = (h.groupby(["sub_theme_name", "variable", "units"], dropna=False)
               .agg(n=("name", "size"), años_sueltos=("anual", "sum"),
                    periodos=("year", lambda s: sorted({str(y) for y in s if "-" in str(y)})),
                    ejemplo=("name", "first"))
               .reset_index())
pd.set_option("display.max_rows", 300, "display.width", 250, "display.max_colwidth", 90)
display(resumen_h)
print("modelos:", h["model"].value_counts().to_dict())

rasters históricos en res01: 3249


,sub_theme_name,variable,units,n,años_sueltos,periodos,ejemplo
0,Climate Classification,Classification by Thermal Climates and Thermal Zones,Class,4,0,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",mc3_CRUTS32_Hist_0010
1,Climate Classification,Koeppen-Geiger (2-character) classification,Class,4,0,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",KG2_CRUTS32_Hist_0010
2,Climate Classification,Koeppen-Geiger (3-character) classification,Class,4,0,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",KG3_CRUTS32_Hist_0010
3,Climate Classification,Multi-cropping class (rain-fed),Class,55,51,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",mcr_CRUTS32_Hist_0010
4,Climate Classification,Multi-cropping class (with irrigation),Class,55,51,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",mci_CRUTS32_Hist_0010
5,Climate Classification,Permafrost zones (classes),Class,4,0,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",pfr_CRUTS32_Hist_0010
6,Climate Classification,Thermal Climates (temperatures reduced to sea level),Class,4,0,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",mcl_CRUTS32_Hist_0010
7,Climate Classification,Thermal Zones,Class,4,0,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",mc2_CRUTS32_Hist_0010
8,Climate Classification,Total number of growing period days class,Class,55,51,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",lgp_CRUTS32_Hist_0010
9,Growing Period,Adjusted LGP for evaluating AgClim Constraints (days),days,55,51,"[1961-1990, 1971-2000, 1981-2010, 2000-2010]",lac_CRUTS32_Hist_0010


modelos: {'CRUTS32': 3249}


### 6b. Construcción de las tablas (tras ver la salida de 6a)
Define el servicio y el filtro `where` (sintaxis SQL sobre los campos del catálogo) de cada tema.
Con `None` la tabla se omite. Si el filtro devuelve más de `TOPE_RASTERS` rasters, no ejecuta: afina el filtro.

In [26]:

# ============================================================
# TEMA 2: GAEZ GROWING (variables agroclimáticas)
# ============================================================
SERV_T2 = "res01"

WHERE_T2 = (
    "rcp = 'Historical' "
    "AND model = 'CRUTS32' "
    "AND year = '1981-2010' "
    "AND sub_theme_name IN ("
        "'Growing Period', "
        "'Moisture Regime', "
        "'Thermal Regime'"
    ") "
    "AND units <> 'Class'"
)

# ============================================================
# TEMA 4: GAEZ SUITABILITY (se mantiene igual)
# ============================================================
SERV_T4 = "res05"

WHERE_T4 = (
    "rcp = 'Historical' "
    "AND model = 'CRUTS32' "
    "AND year = '1981-2010' "
    "AND ("
        "variable LIKE 'Suitability index%current cropland%' OR "
        "variable LIKE 'Average attainable yield of current cropland%' OR "
        "variable LIKE 'Water deficit/net irrigation requirement%current cropland%'"
    ")"
)

TOPE_RASTERS = 3000


In [27]:

# ============================================================
# PREVISUALIZACIÓN DE LOS DOS TEMAS
# ============================================================

print("\n========== GAEZ GROWING (TEMA 2) ==========")
if WHERE_T2:
    previsualizar(SERV_T2, WHERE_T2, mostrar=30)

print("\n========== GAEZ SUITABILITY (TEMA 4) ==========")
if WHERE_T4:
    previsualizar(SERV_T4, WHERE_T4)

# ============================================================
# CONSTRUCCIÓN DE TABLAS
# ============================================================

construir_gaez(
    SERV_T2,
    WHERE_T2,
    "gaez_growing",
    "GAEZ_T2"
)

construir_gaez(
    SERV_T4,
    WHERE_T4,
    "gaez_suitability",
    "GAEZ_T4"
)



========== GAEZ GROWING (TEMA 2) ==========
rasters que cumplen el filtro: 43
sub_theme_name: {'Thermal Regime': 17, 'Moisture Regime': 16, 'Growing Period': 10}
variable: {'Snow stock at end of calendar year (mm)': 1, 'Soil moisture condition at end of calendar year (mm)': 1, 'Snow-adjusted cold temperature limit': 1, 'Reference potential evapotranspiration (using AWC=100 mm/m)': 1, 'Reference actual evapotranspiration (using AWC=100 mm/m)': 1, 'Snow-adjusted air frost number': 1, 'Air frost number': 1, 'Adjusted LGP for evaluating AgClim Constraints (days)': 1, 'Beginning date of earliest growing period (day-of-year; for component LGP > 20 days)': 1, 'Length of longest component LGP (days)': 1, 'Total number of LGP days in component LGPs > 20 days (days)': 1, 'Beginning date of longest component LGP (day-of-year)': 1, 'Total number of growing period days': 1, 'Temperature growing period (LGPt=5); number of days with Ta > 5 °C': 1, 'Temperature growing period (LGPt=10); number of day

,cell_id,lat,lon,valor,zona,objectid,name,category,sub_theme_name,variable,year,model,rcp,units
88,764981,42.875,-94.625,19.000000,iowa,223224,Sb_CRUTS32_Hist_8110,1,Moisture Regime,Snow stock at end of calendar year (mm),1981-2010,CRUTS32,Historical,mm
89,764982,42.875,-94.375,18.444444,iowa,223224,Sb_CRUTS32_Hist_8110,1,Moisture Regime,Snow stock at end of calendar year (mm),1981-2010,CRUTS32,Historical,mm
90,764983,42.875,-94.125,20.000000,iowa,223224,Sb_CRUTS32_Hist_8110,1,Moisture Regime,Snow stock at end of calendar year (mm),1981-2010,CRUTS32,Historical,mm
91,764984,42.875,-93.875,20.444444,iowa,223224,Sb_CRUTS32_Hist_8110,1,Moisture Regime,Snow stock at end of calendar year (mm),1981-2010,CRUTS32,Historical,mm
92,764985,42.875,-93.625,21.333333,iowa,223224,Sb_CRUTS32_Hist_8110,1,Moisture Regime,Snow stock at end of calendar year (mm),1981-2010,CRUTS32,Historical,mm


gaez_suitability ya existe


### 6c. Normalización y control de calidad de `gaez_suitability`
Añade columnas legibles (indicador, régimen de agua, unidades estándar) y verifica que no se colaron valores de "sin dato".
Es idempotente: si ya se aplicó, no hace nada.

In [28]:
if hecho("gaez_suitability"):
    gs = cargar("gaez_suitability")
    if "indicador" not in gs.columns:
        v = gs["variable"].str.lower()
        gs["indicador"] = np.select(
            [v.str.startswith("suitability index"), v.str.startswith("average attainable yield"),
             v.str.startswith("water deficit")],
            ["idoneidad_0_10000", "rendimiento_alcanzable", "deficit_hidrico_mm"], default="otro")
        ws = gs["water_supply"].str.strip()
        gs["regimen_agua"] = np.where(ws.str.startswith("Rainfed"), "secano", "riego")
        gs["tipo_riego"] = ws.where(gs["regimen_agua"] == "riego")          # Irrigation / Gravity / Sprinkler / Drip
        u = gs["units"].str.strip()
        gs["valor_std"] = np.where(u == "10kg DW/ha", gs["valor"] * 10, gs["valor"])
        gs["unidad_std"] = u.replace({"10kg DW/ha": "kg DW/ha"})
        guardar(gs, "gaez_suitability")
    print("filas:", len(gs), "| rasters:", gs["objectid"].nunique(), "| cultivos:", gs["crop"].nunique())
    display(gs.groupby(["indicador", "unidad_std"])["valor_std"].describe().round(1))
    print("valores <= -9 (posible nodata colado):", int((gs["valor_std"] <= -9).sum()))

    # QA: un raster, la celda de California, contra los píxeles originales de GAEZ
    try:
        fila = gs[(gs["zona"] == "california_central") & (gs["indicador"] == "idoneidad_0_10000")].iloc[0]
        url = None
        for _, r in registros("res05", [int(fila["objectid"])], campos_ligeros("res05")).iterrows():
            url = r["download_url"]
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_TIMEOUT="30"):
            with rasterio.open(url) as src:
                c0 = int(round((fila["lon"] - RES / 2 - src.transform.c) / src.res[0]))
                r0 = int(round((src.transform.f - (fila["lat"] + RES / 2)) / src.res[1]))
                bloque = src.read(1, window=Window(c0, r0, 3, 3)).astype(float)
        bloque[bloque == src.nodata] = np.nan
        print(fila["name"], "celda", int(fila["cell_id"]), "| nuestro:", round(fila["valor"], 1),
              "| píxeles GAEZ:", np.round(bloque.ravel()), "| media:", round(float(np.nanmean(bloque)), 1))
    except Exception as ex:
        print("QA no disponible:", type(ex).__name__, str(ex)[:150])

filas: 434844 | rasters: 771 | cultivos: 50


count    mean     std   min  25%     50%     75%      max
indicador              unidad_std                                                               
deficit_hidrico_mm     mm            33276.0    85.2   165.8  -1.0  0.0     0.0    98.9   1155.7
idoneidad_0_10000      Index        200784.0  3771.3  3592.6  -1.0  0.0  3333.0  7060.7  10000.0
rendimiento_alcanzable kg DW/ha     179916.0  2613.4  4222.8 -10.0  0.0   898.3  3367.6  35024.4
                       kg lint/ha     3948.0   290.1   390.9  -1.0  0.0   114.1   419.9   1581.0
                       kg oil/ha      7896.0   139.9   451.4  -1.0  0.0     0.0     0.0   2871.3
                       kg sugar/ha    9024.0  2573.0  3420.6  -1.0  0.0  1006.9  3812.4  13476.7

valores <= -9 (posible nodata colado): 508
suLa_alf celda 744709 | nuestro: 2098.2 | píxeles GAEZ: [-1.000e+00 -1.000e+00  5.556e+03 -1.000e+00 -1.000e+00  5.556e+03
 -1.000e+00 -1.000e+00  7.778e+03] | media: 2098.2


## 7. TABLA 10: `crop_traits` (manual; borrador botánico)
Se genera un CSV editable en Drive (`db/crop_traits.csv`). El grupo lo completa y verifica
(FAO, extensión agraria, literatura). Solo se rellenan familia botánica y fijación de nitrógeno
de los cultivos individuales evidentes; **todo lo demás queda vacío a propósito**.
Los ciclos de plagas necesitan una fuente por explorar.

In [29]:
CSV_TRAITS = DB / "crop_traits.csv"
if hecho("crop_traits"):
    print("crop_traits ya existe")
elif not hecho("gaez_actual"):
    print("Primero genera gaez_actual (de ahí salen los cultivos)")
else:
    if CSV_TRAITS.exists():
        ct = pd.read_csv(CSV_TRAITS)
        print("cargado desde", CSV_TRAITS)
    else:
        base = cargar("gaez_actual")[["crop_code", "crop"]].drop_duplicates().sort_values("crop")
        FAMILIA = {"Wheat": "Poaceae", "Barley": "Poaceae", "Maize": "Poaceae", "Sorghum": "Poaceae",
                   "Millet": "Poaceae", "Wetland rice": "Poaceae", "Sugarcane": "Poaceae",
                   "Soybean": "Fabaceae", "Groundnut": "Fabaceae", "Pulses": "Fabaceae",
                   "Sunflower": "Asteraceae", "Rapeseed": "Brassicaceae", "Cotton": "Malvaceae",
                   "Sugarbeet": "Amaranthaceae", "Olive": "Oleaceae", "Oil palm": "Arecaceae",
                   "Banana": "Musaceae", "Cassava": "Euphorbiaceae", "Tobacco": "Solanaceae"}
        FIJADOR = {"Soybean": "si", "Groundnut": "si", "Pulses": "si"}
        ct = base.copy()
        ct["familia_botanica"] = ct["crop"].map(FAMILIA)
        ct["fijador_nitrogeno"] = ct["crop"].map(FIJADOR).fillna(ct["familia_botanica"].notna().map({True: "no", False: None}))
        for c in ("profundidad_raiz", "ciclo_dias", "necesidad_agua", "plagas_comunes", "fuente_dato"):
            ct[c] = None
        ct["estado"] = "borrador"
        ct.to_csv(CSV_TRAITS, index=False)
        print("creado", CSV_TRAITS, "-> edítalo y verifica cada fila")
    guardar(ct, "crop_traits")
    display(ct)

crop_traits ya existe


## 8. TABLA 11: `indicators_monthly` (derivada)
Umbrales de sequía por **convención** (z < -1 notable, z < -2 extrema), no estándar de GLDAS/NLDAS.
`riesgo_helada` y `ventana_inactividad` quedan vacías hasta tener temperatura mínima (p. ej. NASA POWER)
y `gaez_growing`.

In [30]:
if hecho("indicators_monthly"):
    print("indicators_monthly ya existe")
elif not hecho("climate_monthly"):
    print("Primero genera climate_monthly")
else:
    cmx = cargar("climate_monthly")
    pref = (cmx.assign(prio=cmx["fuente"].map({"NLDAS": 0, "GLDAS": 1}).fillna(9))
               .sort_values("prio").drop_duplicates(["cell_id", "año", "mes"]).drop(columns="prio"))
    zcols = [c for c in ("sm_10_40_z", "sm_40_100_z") if c in pref]
    zr = pref[zcols].mean(axis=1) if zcols else pd.Series(np.nan, index=pref.index)
    ind = pref[["zona", "cell_id", "año", "mes", "fuente"]].copy()
    ind["z_humedad_radicular"] = zr
    ind["nivel_sequia"] = np.select([zr < -2, zr < -1], [2.0, 1.0], default=0.0)
    ind.loc[zr.isna(), "nivel_sequia"] = np.nan
    if "balance" in pref:
        ind["balance_mm"] = pref["balance"]
        ind["deficit_hidrico"] = (pref["balance"] < 0).astype(float).where(pref["balance"].notna())
    if "tair_z" in pref:
        ind["z_temperatura"] = pref["tair_z"]
    ind["riesgo_helada"] = np.nan          # pendiente: temperatura mínima
    ind["ventana_inactividad"] = np.nan    # pendiente: gaez_growing (Tema 2)
    guardar(ind, "indicators_monthly")
    display(ind.head())

indicators_monthly ya existe


## 9. Estado final

In [31]:
estado()
if (DB / "coverage.parquet").exists():
    display(cargar("coverage").pivot_table(index="fuente", values="disponible", aggfunc="sum"))


 1. [OK]        cells                      564 filas | Esqueleto: cuadrícula GLDAS 0.25° + zonas piloto
 2. [OK]        coverage                  3948 filas | Derivada: cobertura de fuentes por celda
 3. [OK]        gaez_actual              42897 filas | GAEZ v4 Tema 5 (.rar local)
 4. [OK]        climate_monthly           8772 filas | GLDAS 2.1 (global) y NLDAS-2 (EE. UU.), Earthdata
 5. [OK]        climatology               8772 filas | Derivada de climate_monthly
 6. [OK]        landcover                 7110 filas | NLCD (MRLC vía pygeohydro), solo EE. UU.
 7. [OK]        stations_validation       1118 filas | CoCoRaHS (CSV) + climate_monthly
 8. [OK]        gaez_growing             23662 filas | GAEZ v4 Tema 2 (módulo del visor)
 9. [OK]        gaez_suitability        434844 filas | GAEZ v4 Tema 4 (módulo del visor)
10. [OK]        crop_traits                 26 filas | Manual (borrador) + fuentes de plagas por explorar
11. [OK]        indicators_monthly        6768 filas | Deriva

,disponible
fuente,
CoCoRaHS,15
GAEZ_T2,564
GAEZ_T4,564
GAEZ_T5,552
GLDAS,563
NLCD,168
NLDAS,168
